In [103]:
import pyspark
from pyspark.sql import SparkSession

In [104]:
"""
Note: The master method sets the master URL for the Spark application.
"local[*]" means that Spark will run locally with as many worker threads as logical cores on your machine. 
This is useful for development and testing purposes.
You can also set it to other cluster managers like yarn, mesos, or provide a Spark standalone cluster URL.
"""
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [72]:
#Uncomment when required (note the files won't be committed due to the .gitignore

#!curl -L -O https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
#!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

In [73]:
#Uncomment when required
#!gzip -dc fhvhv_tripdata_2021-01.csv.gz

In [74]:
# word count
!wc -l fhvhv_tripdata_2021-01.csv

11908469 fhvhv_tripdata_2021-01.csv


In [75]:

df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [76]:
df.schema
"""
Note:
Spark is reading the data as strings instead of timestamps or numbers. 
Unlike Pandas, Spark does not infer data types automatically, so everything is treated as a string by default.
The schema shows all fields are classified as string type.

"""

'\nNote:\nSpark is reading the data as strings instead of timestamps or numbers. \nUnlike Pandas, Spark does not infer data types automatically, so everything is treated as a string by default.\nThe schema shows all fields are classified as string type.\n\n'

In [77]:
#top 20 records
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   null|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   null|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   null|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   null|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   null|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [78]:
#top 5 records
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:33:44', dropoff_datetime='2021-01-01 00:49:07', PULocationID='230', DOLocationID='166', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:55:19', dropoff_datetime='2021-01-01 01:18:21', PULocationID='152', DOLocationID='167', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:23:56', dropoff_datetime='2021-01-01 00:38:05', PULocationID='233', DOLocationID='142', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:42:51', dropoff_datetime='2021-01-01 00:45:50', PULocationID='142', DOLocationID='143', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:48:14', dropoff_datetime='2021-01-01 01:08:42', PULocationID='143', DOLocationID='78', SR_Flag=None)]

In [79]:
# Create a new .csv file in the location and name it as "head.csv". Its 1000 rows
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [80]:
!wc -l head.csv

1001 head.csv


In [81]:
# Powerful data structures for data analysis, time series, and statistics 
# Reading this with pandas instead of spark for size of data
import pandas as pd

In [82]:
df_pandas = pd.read_csv('head.csv')

In [83]:
# checking data types in pandas
df_pandas.dtypes

hvfhs_license_num        object
dispatching_base_num     object
pickup_datetime          object
dropoff_datetime         object
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [ ]:
# Using spark to read the schema after creating a data frame from the pandas df we created (1001 rows of head.csv)
# Note: You might have an error indicating incompatibility between pandas and pyspark version of the dataframe conversion
# IGNORE THE ERROR as not relevant
spark.createDataFrame(df_pandas).schema

Integer - 4 bytes
Long - 8 bytes

In [100]:
from pyspark.sql import types

In [101]:
# Defining the schema
"""
Defining the schema:
To properly define a schema for our DataFrame, I will format the inferred schema. 
Spark schemas use StructType, which is a Scala construct, so I need to convert it into Python code.
"""

schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [102]:
# After defining the schema, I need to specify it when reading the CSV file. 
# Adding the schema parameter ensures that Spark correctly interprets the data types.

df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [19]:
"""
Running df.head(10) on the loaded data confirms that timestamps are parsed correctly, 
location IDs are treated as numbers without quotes, 
and SR_Flag remains a nullable string.

Compare with the previous df.head to see results
"""
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DOLocationID=167, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 23, 56), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 38, 5), PULocationID=233, DOLocationID=142, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 42, 51), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 45, 50), PULocationID=142, DOLocationID=143, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_dat

In [20]:
# Note that we cannot see the difference if we use df.show() 
# df.head() helps to see the data type changes instead
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   null|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   null|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   null|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   null|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   null|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [21]:
# To be precise run this to show the new changes:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', StringType(), True)])

In [23]:
# PARTITIONS
"""
Executing df.repartition(24) does not immediately change the DataFrame because repartitioning is lazy. 
The change is applied only when we perform an action, such as saving the DataFrame.
"""

'\nExecuting df.repartition(24) does not immediately change the DataFrame because repartitioning is lazy. \nThe change is applied only when we perform an action, such as saving the DataFrame.\n'

In [24]:
df = df.repartition(24)

In [25]:
df.write.parquet('fhvhv/2021/01/')

In [ ]:
# SPARK DATAFRAMES

In [44]:
df = spark.read.parquet('fhvhv/2021/01/')

In [26]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



SELECT * FROM df WHERE hvfhs_license_num =  HV0003

In [28]:
# Selecting just few columns
df.select("pickup_datetime", "dropoff_datetime", "PULocationID", "DOLocationID").show()


+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-02 11:32:02|2021-01-02 11:42:27|          76|          76|
|2021-01-03 11:50:12|2021-01-03 12:11:06|         234|         238|
|2021-01-03 19:56:57|2021-01-03 20:03:31|           4|         148|
|2021-01-03 16:32:33|2021-01-03 16:53:09|         113|          49|
|2021-01-02 22:20:05|2021-01-02 22:34:28|          90|         148|
|2021-01-04 16:09:32|2021-01-04 16:27:54|          80|         225|
|2021-01-05 10:22:21|2021-01-05 10:31:03|          61|         177|
|2021-01-02 19:04:39|2021-01-02 19:11:24|         177|         225|
|2021-01-04 09:54:02|2021-01-04 09:59:21|         151|         166|
|2021-01-05 12:29:56|2021-01-05 12:40:57|         214|           6|
|2021-01-01 04:00:17|2021-01-01 04:13:41|          76|          35|
|2021-01-01 05:29:30|2021-01-01 05:47:10|       

In [30]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0005|              B02510|2021-01-01 16:08:13|2021-01-01 16:11:57|          37|          36|   null|
|           HV0005|              B02510|2021-01-04 09:14:45|2021-01-04 09:30:04|          89|         165|   null|
|           HV0003|              B02617|2021-01-05 10:50:11|2021-01-05 11:05:40|         161|         238|   null|
|           HV0003|              B02617|2021-01-03 10:21:50|2021-01-03 10:50:41|         225|         231|   null|
|           HV0005|              B02510|2021-01-01 03:53:08|2021-01-01 04:02:54|         243|         244|   null|
|           HV0003|              B02877|2021-01-01 02:02:05|2021-01-01 02:10:12|

In [52]:
# FILTERING DATA
# Filter statement to get only the records where a specific license number matches a given value
# Note we add show() for spark to actually do something >> Action. 
df.select('hvfhs_license_num', 'pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003') \
  .show(5)


+-----------------+-------------------+-------------------+------------+------------+
|hvfhs_license_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-----------------+-------------------+-------------------+------------+------------+
|           HV0003|2021-01-03 19:19:21|2021-01-03 20:01:53|         188|         159|
|           HV0003|2021-01-02 16:05:00|2021-01-02 16:16:49|         261|         232|
|           HV0003|2021-01-03 06:45:06|2021-01-03 07:00:38|         248|         140|
|           HV0003|2021-01-02 19:49:54|2021-01-02 20:02:11|          35|          77|
|           HV0003|2021-01-03 06:23:22|2021-01-03 06:38:46|         218|          76|
+-----------------+-------------------+-------------------+------------+------------+
only showing top 5 rows



In [53]:
# SPARK ACTIONS examples
"""
show(): Displays the DataFrame.
take(5): Retrieves the first five records. Similar to head()
write.csv() or write.parquet() – Triggers execution to write results to storage.
"""

df.take(5)

[Row(hvfhs_license_num='HV0005', dispatching_base_num='B02510', pickup_datetime=datetime.datetime(2021, 1, 1, 16, 8, 13), dropoff_datetime=datetime.datetime(2021, 1, 1, 16, 11, 57), PULocationID=37, DOLocationID=36, SR_Flag=None),
 Row(hvfhs_license_num='HV0005', dispatching_base_num='B02510', pickup_datetime=datetime.datetime(2021, 1, 4, 9, 14, 45), dropoff_datetime=datetime.datetime(2021, 1, 4, 9, 30, 4), PULocationID=89, DOLocationID=165, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02617', pickup_datetime=datetime.datetime(2021, 1, 5, 10, 50, 11), dropoff_datetime=datetime.datetime(2021, 1, 5, 11, 5, 40), PULocationID=161, DOLocationID=238, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02617', pickup_datetime=datetime.datetime(2021, 1, 3, 10, 21, 50), dropoff_datetime=datetime.datetime(2021, 1, 3, 10, 50, 41), PULocationID=225, DOLocationID=231, SR_Flag=None),
 Row(hvfhs_license_num='HV0005', dispatching_base_num='B02510', pickup_d

In [54]:
# FUNCITONS example to create a new column using the "df.withColumn"
"""
So why use Spark instead? Spark is more flexible and provides additional functionality, such as User-Defined Functions (UDFs).
In Spark, we have pyspark.sql.functions, a collection of functions that Spark provides (Built-in functions). 
To use them you need the following:
"""

'\nSo why use Spark instead? Spark is more flexible and provides additional functionality, such as User-Defined Functions (UDFs).\nIn Spark, we have pyspark.sql.functions, a collection of functions that Spark provides (Built-in functions). \nTo use them you need the following:\n'

In [55]:
from pyspark.sql import functions as F

In [56]:
# One useful function is to_date(), which extracts only the date from a datetime column, discarding hours, minutes, and seconds.
# Similar to SQL´s TO_DATE and CAST(AS date)
# Note: If we use a column name that already exists, Spark overwrites it.

df \
    .withColumn("pickup_date", F.to_date(df.pickup_datetime)) \
    .withColumn("dropoff_date", F.to_date(df.dropoff_datetime)) \
    .show(5)    

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|pickup_date|dropoff_date|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+
|           HV0005|              B02510|2021-01-01 16:08:13|2021-01-01 16:11:57|          37|          36|   null| 2021-01-01|  2021-01-01|
|           HV0005|              B02510|2021-01-04 09:14:45|2021-01-04 09:30:04|          89|         165|   null| 2021-01-04|  2021-01-04|
|           HV0003|              B02617|2021-01-05 10:50:11|2021-01-05 11:05:40|         161|         238|   null| 2021-01-05|  2021-01-05|
|           HV0003|              B02617|2021-01-03 10:21:50|2021-01-03 10:50:41|         225|         231|   null| 2021-01-03|  2021-01-03|
|           HV0005| 

In [57]:
df.show(5)

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0005|              B02510|2021-01-01 16:08:13|2021-01-01 16:11:57|          37|          36|   null|
|           HV0005|              B02510|2021-01-04 09:14:45|2021-01-04 09:30:04|          89|         165|   null|
|           HV0003|              B02617|2021-01-05 10:50:11|2021-01-05 11:05:40|         161|         238|   null|
|           HV0003|              B02617|2021-01-03 10:21:50|2021-01-03 10:50:41|         225|         231|   null|
|           HV0005|              B02510|2021-01-01 03:53:08|2021-01-01 04:02:54|         243|         244|   null|
+-----------------+--------------------+-------------------+-------------------+

In [58]:
# User-Defined Functions
"""
Let's say we have a function that performs complex logic, something not easy to express with SQL. 
I.e, it processes a column called dispatching_base_number, 
Extracts the numeric part of the string by removing the first character (base_num[1:]) and converts it to an integer (num).
And it does string formatting based on the logic divisible provided
"""

"""
Expressing this in SQL would be cumbersome, especially as the logic grows more complex with multiple conditions. 
The advantage of implementing this logic in Python is that it can live in a separate module and it can be unit-tested.
"""

'\nExpressing this in SQL would be cumbersome, especially as the logic grows more complex with multiple conditions. \nThe advantage of implementing this logic in Python is that it can live in a separate module and it can be unit-tested.\n'

In [59]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [60]:
#Testing
crazy_stuff('B02884')

's/b44'

In [51]:
# Now, to turn this Python function into a User-Defined Function (UDF) in PySpark:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [99]:
# Now we can use this udf:


df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [62]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,
